In [1]:
# Always pull shared data from `financial_crime` module
%load_ext autoreload
%autoreload 2

In [2]:
random_state=42

In [3]:
import pandas as pd
from imblearn.under_sampling import RandomUnderSampler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.preprocessing import StandardScaler
import pandas as pd
from imblearn.under_sampling import RandomUnderSampler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
from sklearn.ensemble import GradientBoostingClassifier
from collections import Counter
import pickle
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder

In [4]:
from financial_crime.config import PROCESSED_DATA_DIR, MODELS_DIR

2026-08-21 20:31:55.291 | INFO     | financial_crime.config:<module>:11 - PROJ_ROOT path is: C:\Users\caleb\Projects\financial-crime


## Improvements to Feature Engineering

In [5]:
X = pd.read_parquet(f'{PROCESSED_DATA_DIR}/features.parquet')
X

,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,ID,event_timestamp,Amount_Received_USD,Amount_Paid_USD,Account_Same,Bank_Same
0,2022/09/01 00:20,10,8000EBD30,10,8000EBD30,3697.34,US Dollar,3697.34,US Dollar,Reinvestment,2b4da1a3-5177-492a-bbc1-07a0e617c673,2022-09-01 00:20:00,3697.340000,3697.340000,1,1
1,2022/09/01 00:20,3208,8000F4580,1,8000F5340,0.01,US Dollar,0.01,US Dollar,Cheque,b9fe8bfe-2ef9-4efe-a4f5-297c393a7a55,2022-09-01 00:20:00,0.010000,0.010000,0,0
2,2022/09/01 00:00,3209,8000F4670,3209,8000F4670,14675.57,US Dollar,14675.57,US Dollar,Reinvestment,ef4e3bd2-51a5-4793-ad8b-a45c906a0e75,2022-09-01 00:00:00,14675.570000,14675.570000,1,1
3,2022/09/01 00:02,12,8000F5030,12,8000F5030,2806.97,US Dollar,2806.97,US Dollar,Reinvestment,eb4fc403-a056-4cef-95c2-3f08d3491811,2022-09-01 00:02:00,2806.970000,2806.970000,1,1
4,2022/09/01 00:06,10,8000F5200,10,8000F5200,36682.97,US Dollar,36682.97,US Dollar,Reinvestment,eafc3bec-b6b2-46c3-95b1-861d7d15f105,2022-09-01 00:06:00,36682.970000,36682.970000,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5028340,2022/09/10 17:46,25382,80D141A70,219444,810979BC0,13420.60,Euro,13420.60,Euro,Credit Card,f1f8b7a8-ca92-4b1d-a04e-7c1126788522,2022-09-10 17:46:00,13346.786700,13346.786700,0,0
5028341,2022/09/10 17:42,2467,8016EBBD0,241309,810A97640,148.26,Euro,148.26,Euro,Cheque,00ca1a5e-9322-4d65-9225-1d0025c66145,2022-09-10 17:42:00,147.444570,147.444570,0,0
5028342,2022/09/10 17:35,2467,8016EBBD0,241309,810A97640,307.30,Euro,307.30,Euro,Credit Card,db706f86-c192-4474-8566-fc3883a6327d,2022-09-10 17:35:00,305.609850,305.609850,0,0
5028343,2022/09/10 17:50,1916,810B5D930,13345,810B5DC40,863.09,Euro,863.09,Euro,ACH,4c363eb0-b3fd-4719-b413-8ea5153d3228,2022-09-10 17:50:00,858.343005,858.343005,0,0


In [6]:
y = pd.read_parquet(f'{PROCESSED_DATA_DIR}/labels.parquet')['Is Laundering']
y

0          0
1          0
2          0
3          0
4          0
          ..
5028340    0
5028341    0
5028342    0
5028343    0
5028344    0
Name: Is Laundering, Length: 5028345, dtype: int64

In [7]:
categorical_features = [
    'Receiving Currency',
    'Payment Currency',
    'Payment Format',
]

encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
one_hot_encoded = encoder.fit_transform(X[categorical_features])
one_hot_df = pd.DataFrame(one_hot_encoded, columns=encoder.get_feature_names_out(categorical_features))
X_cleaned = pd.concat([X.drop(categorical_features, axis=1), one_hot_df], axis=1)
X_cleaned

,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Amount Paid,ID,event_timestamp,Amount_Received_USD,...,Payment Currency_US Dollar,Payment Currency_Yen,Payment Currency_Yuan,Payment Format_ACH,Payment Format_Bitcoin,Payment Format_Cash,Payment Format_Cheque,Payment Format_Credit Card,Payment Format_Reinvestment,Payment Format_Wire
0,2022/09/01 00:20,10,8000EBD30,10,8000EBD30,3697.34,3697.34,2b4da1a3-5177-492a-bbc1-07a0e617c673,2022-09-01 00:20:00,3697.340000,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,2022/09/01 00:20,3208,8000F4580,1,8000F5340,0.01,0.01,b9fe8bfe-2ef9-4efe-a4f5-297c393a7a55,2022-09-01 00:20:00,0.010000,...,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
2,2022/09/01 00:00,3209,8000F4670,3209,8000F4670,14675.57,14675.57,ef4e3bd2-51a5-4793-ad8b-a45c906a0e75,2022-09-01 00:00:00,14675.570000,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
3,2022/09/01 00:02,12,8000F5030,12,8000F5030,2806.97,2806.97,eb4fc403-a056-4cef-95c2-3f08d3491811,2022-09-01 00:02:00,2806.970000,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
4,2022/09/01 00:06,10,8000F5200,10,8000F5200,36682.97,36682.97,eafc3bec-b6b2-46c3-95b1-861d7d15f105,2022-09-01 00:06:00,36682.970000,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5028340,2022/09/10 17:46,25382,80D141A70,219444,810979BC0,13420.60,13420.60,f1f8b7a8-ca92-4b1d-a04e-7c1126788522,2022-09-10 17:46:00,13346.786700,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
5028341,2022/09/10 17:42,2467,8016EBBD0,241309,810A97640,148.26,148.26,00ca1a5e-9322-4d65-9225-1d0025c66145,2022-09-10 17:42:00,147.444570,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
5028342,2022/09/10 17:35,2467,8016EBBD0,241309,810A97640,307.30,307.30,db706f86-c192-4474-8566-fc3883a6327d,2022-09-10 17:35:00,305.609850,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
5028343,2022/09/10 17:50,1916,810B5D930,13345,810B5DC40,863.09,863.09,4c363eb0-b3fd-4719-b413-8ea5153d3228,2022-09-10 17:50:00,858.343005,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0


### Has Account sent money to Account.1 before?
For each transaction, sweep all other transactions for that account occuring **before** it. Find whether the account sent money to account.1 before

In [8]:
# Sort features and labels together so each transaction keeps its target.
sort_order = X_cleaned['event_timestamp'].sort_values().index
X_cleaned = X_cleaned.loc[sort_order].reset_index(drop=True)
y = y.loc[sort_order].reset_index(drop=True)

X_cleaned['past_occurrences'] = X_cleaned.groupby(['Account', 'Account.1']).cumcount()
X_cleaned['Account_Transacted_With_Account1_Before'] = (X_cleaned['past_occurrences'] > 0).astype(int)
X_cleaned = X_cleaned.drop(columns=['past_occurrences'])

In [9]:
"""
We have to be careful of data leakage now for training/testing

January: Account A sends money to Account B for the first time. (has_sent_before = False)
February: Account A sends money to Account B again. (has_sent_before = True)
March: Account A sends money to Account B a third time. (has_sent_before = True)

If you use a Random Split:

A random splitter might put the January transaction into your test set, and the February/March transactions into your training set.

When calculating your feature engineering aggregates across the whole dataset before splitting, the dataset records that Account A has sent money to Account B.
 If that "future" knowledge bleeds into the January test row, the model sees has_sent_before = True for January.
 
The Reality: In January, the model should have seen False.
The Consequence: Your model will look incredibly accurate during testing, but it will fail completely in production because it cannot see the future.
"""

'\nWe have to be careful of data leakage now for training/testing\n\nJanuary: Account A sends money to Account B for the first time. (has_sent_before = False)\nFebruary: Account A sends money to Account B again. (has_sent_before = True)\nMarch: Account A sends money to Account B a third time. (has_sent_before = True)\n\nIf you use a Random Split:\n\nA random splitter might put the January transaction into your test set, and the February/March transactions into your training set.\n\nWhen calculating your feature engineering aggregates across the whole dataset before splitting, the dataset records that Account A has sent money to Account B.\n If that "future" knowledge bleeds into the January test row, the model sees has_sent_before = True for January.\n\nThe Reality: In January, the model should have seen False.\nThe Consequence: Your model will look incredibly accurate during testing, but it will fail completely in production because it cannot see the future.\n'

In [10]:
X_cleaned

,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Amount Paid,ID,event_timestamp,Amount_Received_USD,...,Payment Currency_Yen,Payment Currency_Yuan,Payment Format_ACH,Payment Format_Bitcoin,Payment Format_Cash,Payment Format_Cheque,Payment Format_Credit Card,Payment Format_Reinvestment,Payment Format_Wire,Account_Transacted_With_Account1_Before
0,2022/09/01 00:00,1522,800F116A0,1522,800F116A0,65272.09,65272.09,c1e2ef97-fe85-4a2d-8417-83324f45104e,2022-09-01 00:00:00,6.491309e+04,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0
1,2022/09/01 00:00,22,800D856D0,22,800D856D0,6007.61,6007.61,d6f4144b-26bd-4d9a-8b8e-bc02db768afa,2022-09-01 00:00:00,5.974568e+03,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0
2,2022/09/01 00:00,1729,8008C1090,23842,801A97310,13245.37,13245.37,a2ba88af-b23f-40cd-8537-c07d1fdda37a,2022-09-01 00:00:00,1.324537e+04,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0
3,2022/09/01 00:00,228101,80A464B90,228101,80A464B90,17642126.68,17642126.68,b72e6d76-982f-4df8-9ea5-c8826e795aa2,2022-09-01 00:00:00,1.197724e+07,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0
4,2022/09/01 00:00,1665,800D73230,1665,800D73230,17.25,17.25,24d40162-4af3-4072-8b81-79b8834a8523,2022-09-01 00:00:00,1.715513e+01,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5028340,2022/09/18 10:02,9371,8043A0FB0,16163,802F78670,3749.14,3749.14,f514afef-6a6f-4499-9174-bde1eb3de68e,2022-09-18 10:02:00,3.749140e+03,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0
5028341,2022/09/18 11:18,9371,8043A0FB0,13858,8095526B0,1785.27,1785.27,4754217c-2fde-4590-b991-33b6f26d1222,2022-09-18 11:18:00,1.775451e+03,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0
5028342,2022/09/18 11:18,9371,8043A0FB0,9371,8043A0FB0,1785.27,2091.95,82dbe7b2-7727-484a-a828-654d0e4c98f4,2022-09-18 11:18:00,1.775451e+03,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1
5028343,2022/09/18 12:58,9371,8043A0FB0,1124,8026687E0,2154.54,2154.54,20b1556d-a42d-4797-9348-0900acf4c5a5,2022-09-18 12:58:00,2.154540e+03,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0


In [11]:
cols_to_drop = [
    "ID",
    "event_timestamp",
    "Timestamp",
    "To Bank",
    "From Bank",
    "Account",
    "Account.1",
]

In [12]:
X_cleaned.drop(cols_to_drop, axis=1, inplace=True)

In [13]:
X_cleaned

,Amount Received,Amount Paid,Amount_Received_USD,Amount_Paid_USD,Account_Same,Bank_Same,Receiving Currency_Australian Dollar,Receiving Currency_Bitcoin,Receiving Currency_Brazil Real,Receiving Currency_Canadian Dollar,...,Payment Currency_Yen,Payment Currency_Yuan,Payment Format_ACH,Payment Format_Bitcoin,Payment Format_Cash,Payment Format_Cheque,Payment Format_Credit Card,Payment Format_Reinvestment,Payment Format_Wire,Account_Transacted_With_Account1_Before
0,65272.09,65272.09,6.491309e+04,6.491309e+04,1,1,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0
1,6007.61,6007.61,5.974568e+03,5.974568e+03,1,1,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0
2,13245.37,13245.37,1.324537e+04,1.324537e+04,0,0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0
3,17642126.68,17642126.68,1.197724e+07,1.197724e+07,1,1,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0
4,17.25,17.25,1.715513e+01,1.715513e+01,1,1,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5028340,3749.14,3749.14,3.749140e+03,3.749140e+03,0,0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0
5028341,1785.27,1785.27,1.775451e+03,1.775451e+03,0,0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0
5028342,1785.27,2091.95,1.775451e+03,2.091950e+03,1,1,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1
5028343,2154.54,2154.54,2.154540e+03,2.154540e+03,0,0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0


In [14]:
print("Original dataset shape:", Counter(y))
rus = RandomUnderSampler(sampling_strategy=0.5, random_state=random_state)
X_resampled, y_resampled = rus.fit_resample(X_cleaned, y)

print("Resampled dataset shape:", Counter(y_resampled))

Original dataset shape: Counter({0: 5023312, 1: 5033})
Resampled dataset shape: Counter({0: 10066, 1: 5033})


In [15]:
X_resampled

,Amount Received,Amount Paid,Amount_Received_USD,Amount_Paid_USD,Account_Same,Bank_Same,Receiving Currency_Australian Dollar,Receiving Currency_Bitcoin,Receiving Currency_Brazil Real,Receiving Currency_Canadian Dollar,...,Payment Currency_Yen,Payment Currency_Yuan,Payment Format_ACH,Payment Format_Bitcoin,Payment Format_Cash,Payment Format_Cheque,Payment Format_Credit Card,Payment Format_Reinvestment,Payment Format_Wire,Account_Transacted_With_Account1_Before
1395320,523.24,523.24,153.989532,153.989532,0,0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1
31287,2877.33,2877.33,2877.330000,2877.330000,1,1,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0
723011,318432.62,318432.62,2260.871602,2260.871602,0,0,0.0,0.0,0.0,0.0,...,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0
950305,358048.05,358048.05,2542.141155,2542.141155,1,1,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1
671847,33212.79,33212.79,22548.163131,22548.163131,0,0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5028339,9535.31,9535.31,9535.310000,9535.310000,0,0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0
5028340,3749.14,3749.14,3749.140000,3749.140000,0,0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0
5028341,1785.27,1785.27,1775.451015,1775.451015,0,0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0
5028343,2154.54,2154.54,2154.540000,2154.540000,0,0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0


In [16]:
X_resampled = X_resampled.sort_index()

In [17]:
y_resampled = y_resampled.sort_index()

In [18]:
X_train, X_test, y_train, y_test = train_test_split(X_resampled, 
                                                    y_resampled, 
                                                    test_size=0.2,
                                                    shuffle=False,
                                                    random_state=random_state,
)

In [19]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [20]:
# Parameter tuning
import numpy as np
from sklearn.metrics import f1_score
from sklearn.model_selection import TimeSeriesSplit
import optuna


def objective(trial):
    n_estimators = trial.suggest_int("n_estimators", 10, 300)
    max_depth = trial.suggest_int("max_depth", 2, 8)

    # decision_threshold = trial.suggest_float('decision_threshold', 0.1, 0.9)

    tscv = TimeSeriesSplit(n_splits=3)

    fold_f1_scores = []
    # 3. Time-aware cross-validation loop
    for train_idx, val_idx in tscv.split(X_train):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        
        # Fit model
        model = GradientBoostingClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            random_state=random_state
        )
        model.fit(X_tr, y_tr)

        ## # Predict target probability
        # fraud_probabilities = model.predict_proba(X_val)[:, 1]
        # # Apply the optimized decision threshold for this trial
        # custom_predictions = (fraud_probabilities >= decision_threshold).astype(int)
        preds = model.predict(X_val)
        
        # Calculate validation F1 Score
        # score = f1_score(y_val, custom_predictions, zero_division=0)
        score = f1_score(y_val, preds)
        fold_f1_scores.append(score)
        
    # Return average F1 score to maximize
    return np.mean(fold_f1_scores)



study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=100)

top_5_f1_df = study.trials_dataframe().sort_values("value", ascending=False).head(5)
display(top_5_f1_df)

c:\Users\caleb\Projects\financial-crime\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2026-08-21 20:32:12,564] A new study created in memory with name: no-name-81f7a57d-3d6b-4c4c-82e4-323a1406909b
[I 2026-08-21 20:32:13,631] Trial 0 finished with value: 0.8670631278620059 and parameters: {'n_estimators': 53, 'max_depth': 3}. Best is trial 0 with value: 0.8670631278620059.
[I 2026-08-21 20:32:22,941] Trial 1 finished with value: 0.8320155657446252 and parameters: {'n_estimators': 198, 'max_depth': 8}. Best is trial 0 with value: 0.8670631278620059.
[I 2026-08-21 20:32:25,484] Trial 2 finished with value: 0.865879736717187 and parameters: {'n_estimators': 133, 'max_depth': 3}. Best is trial 0 with value: 0.8670631278620059.
[I 2026-08-21 20:32:27,272] Trial 3 finished with value: 0.8658743895460539 and pa

,number,value,datetime_start,datetime_complete,duration,params_max_depth,params_n_estimators,state
79,79,0.870087,2026-08-21 20:34:43.910155,2026-08-21 20:34:44.751871,0 days 00:00:00.841716,4,35,COMPLETE
89,89,0.869824,2026-08-21 20:34:53.354860,2026-08-21 20:34:54.447423,0 days 00:00:01.092563,4,46,COMPLETE
56,56,0.869642,2026-08-21 20:34:14.192918,2026-08-21 20:34:15.469410,0 days 00:00:01.276492,4,53,COMPLETE
21,21,0.869634,2026-08-21 20:33:11.191713,2026-08-21 20:33:12.241616,0 days 00:00:01.049903,4,44,COMPLETE
66,66,0.869634,2026-08-21 20:34:28.444141,2026-08-21 20:34:29.499966,0 days 00:00:01.055825,4,44,COMPLETE


In [24]:
model = GradientBoostingClassifier(
    n_estimators=35,
    max_depth=4,
    random_state=random_state
)

In [25]:
pipeline = make_pipeline(
    StandardScaler(),
    GradientBoostingClassifier(
        n_estimators=66,
        max_depth=6,
        random_state=random_state
    )
)

In [26]:
pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

print(classification_report(y_true=y_test, y_pred=y_pred))

              precision    recall  f1-score   support

           0       0.91      0.97      0.94      1564
           1       0.96      0.89      0.93      1456

    accuracy                           0.93      3020
   macro avg       0.93      0.93      0.93      3020
weighted avg       0.93      0.93      0.93      3020

